# Displacement Probability Stepwise Debugging

This notebook helps you verify the computations of displacement probabilities step by step.

In [ ]:
# Imports
import numpy as np
import os
import json

os.chdir("/home/ebr/projects/release-volume-sampler/src")


In [ ]:

from rvsampler.utils import read_tif
from rvsampler.displacements import displacement

## 1. Load Cumulative Data

In [ ]:
# Set directories
rundir = "/home/ebr/projects/release-volume-sampler/generated/messina_003"
ky_dir = os.path.join(rundir, "slope_analysis", "yield_acceleration", "cumulative")
pga_dir = os.path.join(rundir, "shakemaps")

# Load content.json for ky
with open(os.path.join(ky_dir, "content.json"), 'r') as f:
    ky_content = json.load(f)
print("Yield acceleration thresholds:", [e["threshold"] for e in ky_content])

In [ ]:
# Load rasters for ky
ky_rasters = []
ky_thresholds = []
for e in ky_content:
    ky_thresholds.append(e["threshold"])
    raster_path = os.path.join(ky_dir, e["file"])
    raster_data, msk, profile = read_tif(raster_path, None)
    ky_rasters.append(raster_data)
ky_rasters = np.stack(ky_rasters)
ky_thresholds = np.array(ky_thresholds)
print("ky_rasters shape:", ky_rasters.shape)

In [ ]:
# Repeat for pga
with open(os.path.join(pga_dir, "content.json"), 'r') as f:
    pga_content = json.load(f)
pga_rasters = []
for e in pga_content:
    raster_path = os.path.join(pga_dir, e["file"])
    raster_data, msk, profile = read_tif(raster_path, None)
    pga_rasters.append(raster_data)
pga_rasters = np.stack(pga_rasters)
print("pga_rasters shape:", pga_rasters.shape)


In [ ]:
nr_of_thresholds = 100
pga_thresholds = np.linspace(start=pga_rasters.min(), stop=pga_rasters.max(), num=nr_of_thresholds)
print("pga_thresholds:", pga_thresholds)

## 2. Compute Densities

In [ ]:
ky_density = np.diff(ky_rasters, axis=0)
print("ky_density shape:", ky_density.shape)

## 3. Compute Displacement at Grid Centers

In [ ]:
ky_centers = 0.5 * (ky_thresholds[1:] + ky_thresholds[:-1])
pga_centers = 0.5 * (pga_thresholds[1:] + pga_thresholds[:-1])
kys, pgas = np.meshgrid(ky_centers, pga_centers)
magnitude = 7.0  # Set your event magnitude here
log_d, log_sigma = displacement(kys, pgas, magnitude)
print("log_d shape:", log_d.shape)

## 4. Compute Probabilities for a Threshold

In [ ]:
delta = 5  # Example threshold in cm
d_is_bigger = log_d > np.log10(delta)

In [ ]:
# Compute conditional probability P(displacement > delta | PGA)
# d_is_bigger: shape (n_pga_bins, n_ky_bins)
# ky_density: shape (n_ky_bins, y_res, x_res)
probs_by_threshold = np.tensordot(ky_density, d_is_bigger, axes=[0,1]) # shape (y_res, x_res, n_pga_bins)
print("probs_by_threshold shape:", probs_by_threshold.shape)

In [ ]:
import matplotlib.pyplot as plt

indices = [0, 50, 98]
for i in indices:
    plt.figure(figsize=(6, 5))
    plt.title(f"Conditional probability heatmap for PGA={pga_thresholds[i]}")
    plt.imshow(probs_by_threshold[:,:,i], aspect='auto', origin='lower')
    plt.colorbar(label="Probability")
    plt.show()

In [ ]:
# Suppose:
# - pga_centers: shape (n_pga_bins,)
# - conditional_probs: shape (n_pga_bins,) or (n_pga_bins, y_res, x_res) if spatially varying

# If conditional_probs is the same for all pixels (1D):
# For each pixel, find the nearest PGA bin and assign the probability

# pga_raster: shape (y_res, x_res)
#pga_raster = pga_rasters[1]

#indices = np.searchsorted(pga_thresholds, pga_raster, side="right") - 1

# Error
# np.mean(np.take(pga_thresholds, indices) - pga_raster)


In [ ]:
# probs_by_threshold: shape (y_res, x_res, n_pga_bins)
# indices: shape (y_res, x_res), integer indices along the last axis

# Add a new axis to indices to match the dimensions for take_along_axis
probs_by_samples = []
for pga_raster in pga_rasters:
    print(pga_raster.shape)
    indices = np.searchsorted(pga_thresholds, pga_raster, side="right") - 1
    indices = np.clip(indices, 0, probs_by_threshold.shape[2] - 1)

    y_res, x_res, n_pga_bins= probs_by_threshold.shape
    out = np.zeros((y_res, x_res))
    for ii in range(x_res):
        for jj in range(y_res):
            out[jj, ii] = probs_by_threshold[jj, ii, indices[jj, ii]]
    probs_by_samples.append(out)


In [ ]:
# Vectorized verision  
# mainly the vectorization of the inner for loop that speeds up the computation.

# pga_rasters: shape (n_samples, y_res, x_res)
# probs_by_threshold: shape (y_res, x_res, n_pga_bins)
n_samples, y_res, x_res = pga_rasters.shape

# Compute indices for all samples at once
indices = np.searchsorted(pga_thresholds, pga_rasters, side="right") - 1  # shape: (n_samples, y_res, x_res)
indices = np.clip(indices, 0, probs_by_threshold.shape[2] - 1)

# Expand probs_by_threshold to broadcast over samples
probs_by_threshold_broadcasted = np.broadcast_to(probs_by_threshold, (n_samples, y_res, x_res, probs_by_threshold.shape[2]))

# Prepare indices for take_along_axis
indices_expanded = indices[..., None]  # shape: (n_samples, y_res, x_res, 1)

# Vectorized lookup
probs_by_samples = np.take_along_axis(probs_by_threshold_broadcasted, indices_expanded, axis=3)[..., 0]  # shape: (n_samples, y_res, x_res)

In [ ]:
for sample in range(10):
    plt.figure(figsize=(6, 5))
    plt.title(f"Exceedance probability heatmap for sample={sample}")
    plt.imshow(probs_by_samples[sample], aspect='auto', origin='lower')
    plt.colorbar(label="Probability")
    plt.show()

In [ ]:
for i,p in enumerate(probs_by_samples):
    print(i)
    print(p.shape)